# Classification DatScan — Pipeline complet (crop physique + normalisation + entraînement)

Ce notebook part du zip brut de volumes `.nii.gz` et couvre tout le pipeline.

**Historique des décisions prises en amont** (pour mémoire) :
- Le dataset mélange plusieurs protocoles/résolutions très différents (FOV allant de ~175mm
  à ~630mm, résolutions spatiales variées, quantification d'intensité très hétérogène).
- Un recalage affine vers un template T1 (MNI152) a été testé mais **abandonné** : le DatScan
  a trop peu de structure anatomique visible en dehors du striatum pour que le recalage par
  intensité s'accroche correctement — résultat en rotations aberrantes.
- Approche retenue : un **crop centré sur une taille physique fixe (en mm)**, dérivée du
  spacing de chaque image individuellement, puis resize vers une shape fixe en voxels — plus
  simple, plus robuste, pas de dépendance à un template externe. Complété par une
  normalisation d'intensité **par image** (percentile 1-99), pour absorber l'hétérogénéité de
  quantification observée entre sources.
- Cette approche ne corrige pas d'éventuelles différences de rotation/orientation entre
  scans — on compense partiellement par de l'augmentation (flips) à l'entraînement.


## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install -q nibabel monai tqdm scikit-learn

## 1. Configuration

In [ ]:
DATA_DIR = "/content/drive/MyDrive/Stats/Parkinson"
OUTER_ZIP = f"{DATA_DIR}/<nom_du_zip>.zip"              # <-- adapte le nom
LABELS_CSV = f"{DATA_DIR}/<nom_du_fichier_labels>.csv"  # <-- adapte le nom
OUTPUT_DIR = f"{DATA_DIR}/outputs"

FINAL_DIR = f"{DATA_DIR}/final_v2"
PREP_LOG_PATH = f"{OUTPUT_DIR}/prep_log.csv"

RANDOM_SEED = 42
PHYSICAL_SIZE_MM = 200       # taille du crop, en mm, avant resize
IMG_SIZE = (96, 96, 96)      # taille d'entrée du modèle
BATCH_SIZE = 2
N_EPOCHS_CNN = 20
N_EPOCHS_SWIN = 30

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FINAL_DIR, exist_ok=True)

## 2. Imports

In [ ]:
import os
import csv
import zipfile
import random

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from monai.networks.nets import SwinUNETR
from monai.losses import FocalLoss

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

## 3. Fonctions de prétraitement

Crop centré sur le contenu, à une taille physique fixe (en mm) — pour rendre les volumes
comparables entre protocoles malgré des FOV d'origine très différents — puis resize vers
une shape fixe en voxels, et normalisation d'intensité par percentile (par image).

In [ ]:
def load_nifti_from_zip(zip_path, member_name, tmp_dir="/content/_tmp_nii"):
    os.makedirs(tmp_dir, exist_ok=True)
    tmp_path = os.path.join(tmp_dir, os.path.basename(member_name))
    with zipfile.ZipFile(zip_path) as z:
        with z.open(member_name) as src, open(tmp_path, "wb") as dst:
            dst.write(src.read())
    return nib.load(tmp_path)


def crop_to_fixed_physical_size(img, physical_size_mm=200):
    """Recadre un volume au centre du signal, sur une taille physique fixe (en mm),
    peu importe le FOV d'origine."""
    data = img.get_fdata().astype(np.float32)
    voxel_size = np.array(img.header.get_zooms()[:3])

    threshold = np.percentile(data, 1) + 1e-6
    mask = data > threshold
    if not mask.any():
        center = np.array(data.shape) // 2
    else:
        coords = np.array(np.nonzero(mask))
        center = coords.mean(axis=1).astype(int)

    crop_size_voxels = (physical_size_mm / voxel_size).astype(int)

    crop_slices = []
    for axis in range(3):
        half = crop_size_voxels[axis] // 2
        start = max(center[axis] - half, 0)
        end = min(center[axis] + half, data.shape[axis])
        crop_slices.append(slice(start, end))

    return data[tuple(crop_slices)]


def center_crop_or_pad(data, target_shape):
    """Recadre ou complète (pad) un volume 3D pour atteindre exactement target_shape,
    centré sur les axes."""
    crop_slices = []
    for axis in range(3):
        size = data.shape[axis]
        target = target_shape[axis]
        if size > target:
            start = (size - target) // 2
            crop_slices.append(slice(start, start + target))
        else:
            crop_slices.append(slice(0, size))
    cropped = data[tuple(crop_slices)]

    pad_widths = []
    for axis in range(3):
        size = cropped.shape[axis]
        target = target_shape[axis]
        total_pad = max(target - size, 0)
        pad_before = total_pad // 2
        pad_after = total_pad - pad_before
        pad_widths.append((pad_before, pad_after))

    return np.pad(cropped, pad_widths, mode="constant", constant_values=0)


def preprocess_volume(img, physical_size_mm=200, target_shape=(128, 128, 128)):
    """Pipeline complet : crop physique -> crop/pad vers shape fixe -> normalisation
    d'intensité par percentile (propre à cette image)."""
    cropped = crop_to_fixed_physical_size(img, physical_size_mm=physical_size_mm)
    final = center_crop_or_pad(cropped, target_shape)

    vmin, vmax = np.percentile(final, [1, 99])
    final_norm = np.clip((final - vmin) / (vmax - vmin + 1e-6), 0, 1)

    return final_norm.astype(np.float32)

## 4. Test visuel sur quelques fichiers avant le batch complet

Reprend les fichiers représentatifs des différents groupes de FOV identifiés précédemment.

In [ ]:
test_files = ["04x0qzfh.nii.gz", "0224wk0y.nii.gz", "085rntdr.nii.gz"]  # adapte si besoin

fig, axes = plt.subplots(len(test_files), 3, figsize=(9, 3 * len(test_files)))

for row, name in enumerate(test_files):
    img = load_nifti_from_zip(OUTER_ZIP, name)
    result = preprocess_volume(img, physical_size_mm=PHYSICAL_SIZE_MM, target_shape=(128, 128, 128))
    x, y, z = result.shape

    axes[row][0].imshow(result[x // 2, :, :].T, cmap="gray", origin="lower", vmin=0, vmax=1)
    axes[row][1].imshow(result[:, y // 2, :].T, cmap="gray", origin="lower", vmin=0, vmax=1)
    axes[row][2].imshow(result[:, :, z // 2].T, cmap="gray", origin="lower", vmin=0, vmax=1)
    axes[row][0].set_ylabel(name, fontsize=8)
    for ax in axes[row]:
        ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.show()

## 5. Batch de prétraitement complet

Log incrémental — reprenable en cas d'interruption.

In [ ]:
def batch_preprocess(zip_path, output_dir, log_path, physical_size_mm=200, target_shape=(128, 128, 128), limit=None):
    os.makedirs(output_dir, exist_ok=True)

    with zipfile.ZipFile(zip_path) as z:
        members = [n for n in z.namelist() if n.endswith(".nii.gz")]
    if limit is not None:
        members = members[:limit]

    already_done = set()
    if os.path.exists(log_path):
        already_done = set(pd.read_csv(log_path)["filename"])
    else:
        with open(log_path, "w", newline="") as f:
            csv.writer(f).writerow(["filename", "status", "error"])

    for name in tqdm(members, desc="Prétraitement"):
        if name in already_done:
            continue

        out_path = os.path.join(output_dir, os.path.basename(name))
        row = {"filename": name, "status": None, "error": None}

        if os.path.exists(out_path):
            row["status"] = "skipped_existing"
        else:
            try:
                img = load_nifti_from_zip(zip_path, name)
                result = preprocess_volume(img, physical_size_mm=physical_size_mm, target_shape=target_shape)
                out_img = nib.Nifti1Image(result, np.eye(4))
                nib.save(out_img, out_path)
                row["status"] = "ok"
            except Exception as e:
                row["status"] = "error"
                row["error"] = str(e)

        with open(log_path, "a", newline="") as f:
            csv.writer(f).writerow([row["filename"], row["status"], row["error"]])

    return pd.read_csv(log_path)

In [ ]:
prep_log = batch_preprocess(OUTER_ZIP, FINAL_DIR, PREP_LOG_PATH,
                             physical_size_mm=PHYSICAL_SIZE_MM, target_shape=(128, 128, 128),
                             limit=None)
print(prep_log["status"].value_counts())

## 6. Labels et split train/val

In [ ]:
labels_df = pd.read_csv(LABELS_CSV)
labels_df = labels_df.dropna(subset=["is_pathologic"])
print(labels_df.shape)

In [ ]:
available_uids = set(
    f.replace(".nii.gz", "") for f in os.listdir(FINAL_DIR) if f.endswith(".nii.gz")
)
print(f"{len(available_uids)} fichiers prétraités disponibles")

labels_df_filtered = labels_df[labels_df["uid"].isin(available_uids)].reset_index(drop=True)
print(f"{len(labels_df_filtered)} labels utilisables après filtrage")

filepaths = [os.path.join(FINAL_DIR, f"{uid}.nii.gz") for uid in labels_df_filtered["uid"]]
labels = labels_df_filtered["is_pathologic"].astype(int).values

train_paths, val_paths, train_labels, val_labels = train_test_split(
    filepaths, labels, test_size=0.2, stratify=labels, random_state=RANDOM_SEED
)

print(f"Train: {len(train_paths)}, Val: {len(val_paths)}")
print("Répartition train:", np.bincount(train_labels))
print("Répartition val:", np.bincount(val_labels))

## 7. Dataset et DataLoader

Volumes déjà normalisés (0-1) et à shape fixe (128³) — resize vers `IMG_SIZE`, et
augmentation légère (flip, bruit) pour compenser en partie l'absence de correction
d'orientation entre scans.

In [ ]:
def resize_volume(volume, target_shape):
    t = torch.from_numpy(volume).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=target_shape, mode="trilinear", align_corners=False)
    return t.squeeze(0)


class DatScanDataset(Dataset):
    def __init__(self, filepaths, labels, target_shape=IMG_SIZE, augment=False):
        self.filepaths = filepaths
        self.labels = labels
        self.target_shape = target_shape
        self.augment = augment

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        data = nib.load(self.filepaths[idx]).get_fdata().astype(np.float32)
        img = resize_volume(data, self.target_shape)

        if self.augment:
            if random.random() < 0.5:
                img = torch.flip(img, dims=[1])
            if random.random() < 0.2:
                img = img + torch.randn_like(img) * 0.01

        label = self.labels[idx]
        return img, torch.tensor(label, dtype=torch.long)

In [ ]:
train_ds = DatScanDataset(train_paths, train_labels, augment=True)
val_ds = DatScanDataset(val_paths, val_labels, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

sample_img, sample_label = train_ds[0]
print("Shape d'un échantillon:", sample_img.shape, "— label:", sample_label)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 8. Diagnostic visuel — quelques exemples de chaque classe

À vérifier avant d'investir du temps d'entraînement : le signal doit être au moins
visuellement perceptible.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

class0_uids = labels_df_filtered[labels_df_filtered["is_pathologic"] == 0]["uid"].values[:4]
class1_uids = labels_df_filtered[labels_df_filtered["is_pathologic"] == 1]["uid"].values[:4]

for col, uid in enumerate(class0_uids):
    img = nib.load(os.path.join(FINAL_DIR, f"{uid}.nii.gz")).get_fdata()
    z = img.shape[2] // 2
    axes[0][col].imshow(img[:, :, z].T, cmap="gray", origin="lower", vmin=0, vmax=1)
    axes[0][col].set_title(f"Classe 0 — {uid}", fontsize=8)
    axes[0][col].axis("off")

for col, uid in enumerate(class1_uids):
    img = nib.load(os.path.join(FINAL_DIR, f"{uid}.nii.gz")).get_fdata()
    z = img.shape[2] // 2
    axes[1][col].imshow(img[:, :, z].T, cmap="gray", origin="lower", vmin=0, vmax=1)
    axes[1][col].set_title(f"Classe 1 — {uid}", fontsize=8)
    axes[1][col].axis("off")

plt.tight_layout()
plt.show()

## 9. Baseline — CNN 3D léger

In [ ]:
class SimpleCNN3D(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm3d(16), nn.ReLU(), nn.MaxPool3d(2),

            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32), nn.ReLU(), nn.MaxPool3d(2),

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64), nn.ReLU(), nn.MaxPool3d(2),

            nn.AdaptiveAvgPool3d(1),
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

In [ ]:
criterion = FocalLoss(gamma=2.0, to_onehot_y=True, use_softmax=True)

def train_model(model, optimizer, train_loader, val_loader, n_epochs, checkpoint_path):
    best_val_acc = 0.0
    history = []

    for epoch in range(n_epochs):
        model.train()
        train_loss = 0
        for imgs, labels_batch in train_loader:
            imgs, labels_batch = imgs.to(device), labels_batch.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels_batch.unsqueeze(1))
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels_batch in val_loader:
                imgs, labels_batch = imgs.to(device), labels_batch.to(device)
                outputs = model(imgs)
                preds = outputs.argmax(dim=1)
                correct += (preds == labels_batch).sum().item()
                total += labels_batch.size(0)

        val_acc = correct / total
        avg_train_loss = train_loss / len(train_loader)
        history.append({"epoch": epoch + 1, "train_loss": avg_train_loss, "val_acc": val_acc})
        print(f"Epoch {epoch+1}/{n_epochs} — train loss: {avg_train_loss:.4f} — val acc: {val_acc:.4f}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), checkpoint_path)
            print(f"  -> Nouveau meilleur modèle sauvegardé (val acc: {val_acc:.4f})")

    return pd.DataFrame(history)

In [ ]:
cnn_model = SimpleCNN3D(num_classes=2).to(device)
cnn_optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=1e-3, weight_decay=1e-5)

cnn_history = train_model(
    cnn_model, cnn_optimizer, train_loader, val_loader,
    n_epochs=N_EPOCHS_CNN,
    checkpoint_path=f"{OUTPUT_DIR}/best_cnn_model.pt",
)

## 10. Swin Transformer 3D

À lancer seulement si le CNN baseline montre un signal exploitable (val accuracy nettement
au-dessus de la proportion de la classe majoritaire, avec tendance à la hausse).

In [ ]:
class SwinClassifier(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        backbone = SwinUNETR(
            in_channels=in_channels,
            out_channels=num_classes,
            feature_size=48,
        )
        self.encoder = backbone.swinViT
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.head = nn.Linear(768, num_classes)

    def forward(self, x):
        features = self.encoder(x)[-1]
        pooled = self.pool(features).flatten(1)
        return self.head(pooled)

In [ ]:
swin_model = SwinClassifier(num_classes=2).to(device)
swin_optimizer = torch.optim.AdamW(swin_model.parameters(), lr=1e-4, weight_decay=1e-5)

swin_history = train_model(
    swin_model, swin_optimizer, train_loader, val_loader,
    n_epochs=N_EPOCHS_SWIN,
    checkpoint_path=f"{OUTPUT_DIR}/best_swin_model.pt",
)

## 11. Évaluation détaillée

L'accuracy seule est trompeuse vu le déséquilibre de classes (~55/45).

In [ ]:
def evaluate_model(model, val_loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, labels_batch in val_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = outputs.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels_batch.numpy())
            all_probs.extend(probs.cpu().numpy())

    print("Matrice de confusion:")
    print(confusion_matrix(all_labels, all_preds))
    print("\nRapport de classification:")
    print(classification_report(all_labels, all_preds, target_names=["sain", "pathologique"]))
    try:
        auc = roc_auc_score(all_labels, all_probs)
        print(f"AUC: {auc:.4f}")
    except ValueError:
        print("AUC non calculable (une seule classe présente dans les prédictions)")

    return all_preds, all_labels, all_probs

In [ ]:
print("=== CNN baseline ===")
cnn_model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best_cnn_model.pt"))
_ = evaluate_model(cnn_model, val_loader)

In [ ]:
print("=== Swin Transformer ===")
swin_model.load_state_dict(torch.load(f"{OUTPUT_DIR}/best_swin_model.pt"))
_ = evaluate_model(swin_model, val_loader)

## 12. Résultats et conclusions

*(à compléter — si le signal reste absent même avec cette approche plus simple, les pistes
restantes sont : un vrai template DatScan pour un recalage adapté, une vérification plus
poussée de la fiabilité des labels, ou une revue avec un expert du domaine des images qui
semblent visuellement ambiguës)*